### Basic data exploration, cleaning, and resampling of iSUPER 2023, 2024, and 2025 (Jan-Aug) dataset

### EDA on dataset

In [72]:
# import libraries
import pandas as pd
import numpy as np

# list of input files (add more file if required)
files = [
    "iSUPER data/Chelsea DEP MOD-00247 Full 2023.csv",
    "iSUPER data/Chelsea DEP MOD-00247 Full 2024.csv",
    "iSUPER data/Chelsea DEP MOD-00214 Jan-Nov 2025.csv",
]

# read and stack all the years together
df_list = []

for file in files:
    df_raw = pd.read_csv(file)
    print(f"Loaded {file} with shape {df_raw.shape}")
    # Rename columns -> because we have different flag name spacing in different files
    rename_dict = {}
    for col in df_raw.columns:
        if col.replace(" ", "").lower() == "pm10>150?":
            rename_dict[col] = "pm10 > 150?"
    if rename_dict:
        df_raw = df_raw.rename(columns=rename_dict)

    df_list.append(df_raw)

# concatenate
df_raw = pd.concat(df_list, ignore_index=True)

display(df_raw[2000:3000])
print(df_raw.dtypes)
df_raw.shape

Loaded iSUPER data/Chelsea DEP MOD-00247 Full 2023.csv with shape (451813, 16)
Loaded iSUPER data/Chelsea DEP MOD-00247 Full 2024.csv with shape (515983, 16)
Loaded iSUPER data/Chelsea DEP MOD-00214 Jan-Nov 2025.csv with shape (373854, 16)


,timestamp,timestamp_local,sn,rh,temp,lat,lon,device_state,pm1,pm25,pm10,co,no,no2,o3,pm10 > 150?
2000,2023-12-29T02:14:04Z,2023-12-28T21:14:04Z,MOD-00247,88.6,7.0,42.387,-71.025,ACTIVE,0.365,0.532,0.777,175.178,4.404,27.742,13.299,0.0
2001,2023-12-29T02:13:04Z,2023-12-28T21:13:04Z,MOD-00247,88.7,7.0,42.387,-71.025,ACTIVE,0.475,0.760,0.772,171.412,4.409,27.738,12.877,0.0
2002,2023-12-29T02:12:04Z,2023-12-28T21:12:04Z,MOD-00247,88.8,7.0,42.387,-71.025,ACTIVE,0.239,0.596,1.315,177.909,4.406,28.207,13.666,0.0
2003,2023-12-29T02:11:04Z,2023-12-28T21:11:04Z,MOD-00247,88.7,7.0,42.387,-71.025,ACTIVE,0.252,0.323,0.583,185.780,4.408,28.685,12.877,0.0
2004,2023-12-29T02:10:04Z,2023-12-28T21:10:04Z,MOD-00247,88.8,7.0,42.387,-71.025,ACTIVE,0.301,0.819,1.340,209.929,4.597,28.444,12.858,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,2023-12-28T09:39:03Z,2023-12-28T04:39:03Z,MOD-00247,89.9,8.0,42.387,-71.025,ACTIVE,2.889,3.077,8.953,247.173,3.968,11.924,6.010,0.0
2996,2023-12-28T09:38:03Z,2023-12-28T04:38:03Z,MOD-00247,89.9,8.0,42.387,-71.025,ACTIVE,2.595,3.038,40.994,237.321,4.332,11.921,5.607,0.0
2997,2023-12-28T09:37:03Z,2023-12-28T04:37:03Z,MOD-00247,89.8,8.0,42.387,-71.025,ACTIVE,2.524,2.766,45.165,241.087,4.332,11.904,7.243,0.0
2998,2023-12-28T09:36:03Z,2023-12-28T04:36:03Z,MOD-00247,89.8,8.0,42.387,-71.025,ACTIVE,3.182,3.697,25.576,249.297,4.329,11.916,6.840,0.0


timestamp           object
timestamp_local     object
sn                  object
rh                 float64
temp               float64
lat                float64
lon                float64
device_state        object
pm1                float64
pm25               float64
pm10               float64
co                 float64
no                 float64
no2                float64
o3                 float64
pm10 > 150?        float64
dtype: object


(1341650, 16)

### Re-sampling and cleaning

We will focus on the some columns in this dataset, hourly sampling is also done (not required for this dataset though but its good to be safe). Missing data is present here, this should be dealt after merging with other datasets.

In [73]:
# rename timestamp_local to timestamp_utc
df_raw["timestamp_utc"] = pd.to_datetime(df_raw["timestamp"], utc=True)

In [74]:
# keep only relevant columns
df_cleaned = df_raw[[
    "timestamp_utc",
    "temp",
    "rh",
    "pm1",
    "pm25",
    "pm10",
    #"pm10 > 150?", # remove due to comment from advisor
]].copy()

display(df_cleaned.head(10))
df_cleaned.shape

,timestamp_utc,temp,rh,pm1,pm25,pm10
0,2023-12-31 23:59:41+00:00,3.3,53.3,3.322,3.344,20.029
1,2023-12-31 23:58:41+00:00,3.3,53.4,3.351,3.455,3.521
2,2023-12-31 23:57:41+00:00,3.3,53.4,3.644,3.683,3.683
3,2023-12-31 23:56:41+00:00,3.3,53.6,3.131,3.211,4.007
4,2023-12-31 23:55:41+00:00,3.3,53.7,3.768,3.931,4.052
5,2023-12-31 23:54:41+00:00,3.3,53.8,3.313,3.445,18.818
6,2023-12-31 23:53:41+00:00,3.3,53.6,4.337,4.392,4.673
7,2023-12-31 23:52:41+00:00,3.3,53.7,4.984,5.041,5.333
8,2023-12-31 23:51:41+00:00,3.3,53.5,4.539,4.551,4.551
9,2023-12-31 23:50:41+00:00,3.3,53.4,4.686,4.802,5.153


(1341650, 6)

In [75]:
df_cleaned = df_cleaned.set_index("timestamp_utc").sort_index()
df_cleaned.head(5)

,temp,rh,pm1,pm25,pm10
timestamp_utc,,,,,
2023-02-14 20:08:34+00:00,16.1,18.5,1.896,2.453,19.011
2023-02-14 20:09:34+00:00,16.0,18.6,2.205,2.561,20.050
2023-02-14 20:10:34+00:00,15.9,18.8,2.217,2.452,2.656
2023-02-14 20:11:34+00:00,15.8,18.7,2.216,2.432,12.210
2023-02-14 20:12:34+00:00,15.8,19.0,2.099,2.604,11.426


In [76]:
# In this dataset, out data are already hourly, but we can still resample to enforce a clean hourly format. 
# Keep it consistent with other datasets cleaning script.
agg_rules = {
    "rh": "mean",
    "temp": "mean",
    "pm1": "mean",
    "pm25": "mean",
    "pm10": "mean",
    #"pm10 > 150?": "max", # Here, we use max so that "any exceedance within an hour" would be flag as "1"
}

isuper_5min = df_cleaned.resample("5T").agg(agg_rules)

C:\Users\USER\AppData\Local\Temp\ipykernel_75304\3443841788.py:12: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  isuper_5min = df_cleaned.resample("5T").agg(agg_rules)


In [77]:
isuper_5min = isuper_5min.rename(columns={"temp": "temp_sensor", "rh": "rh_sensor"})

In [78]:
isuper_5min.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 288911 entries, 2023-02-14 20:05:00+00:00 to 2025-11-13 23:55:00+00:00
Freq: 5min
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   rh_sensor    268723 non-null  float64
 1   temp_sensor  268723 non-null  float64
 2   pm1          268361 non-null  float64
 3   pm25         268361 non-null  float64
 4   pm10         268361 non-null  float64
dtypes: float64(5)
memory usage: 13.2 MB


In [79]:
# save csv file as isuper_hourly
isuper_5min.to_csv("Filtered dataset/isuper_5min_full.csv")